# PRCP-1016: Heart Disease Prediction

**Domain:** Healthcare  |  **Goal:** predict whether a patient has heart disease from 13 clinical measurements.

| Task | Where in this notebook |
|---|---|
| Task 1 - Complete data analysis report | Sections 1-3 |
| Task 2 - ML model to predict heart disease | Sections 4-6 |
| Task 3 - Suggestions to the hospital | Section 8 |
| Model Comparison Report | Section 7 |
| Report on Challenges Faced | Section 9 |

**Context.** Cardiovascular diseases are the number one cause of death worldwide (about 17.9 million deaths a year). Early detection in high-risk people is where a machine learning model can help most. Because a missed patient is far more costly than a false alarm, **recall (sensitivity)** is treated as the key metric alongside ROC-AUC.

## 1. Data loading and overview
The data comes as two files: `values.csv` (patient features) and `labels.csv` (target `heart_disease_present`). They are joined on `patient_id`.

In [1]:
 
import warnings, os
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt, seaborn as sns
from scipy import stats
from sklearn.model_selection import (train_test_split, StratifiedKFold, RepeatedStratifiedKFold,
                                     cross_validate, GridSearchCV, cross_val_predict)
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier, ExtraTreesClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
                             confusion_matrix, classification_report, RocCurveDisplay, roc_curve,
                             ConfusionMatrixDisplay)
from sklearn.inspection import permutation_importance
sns.set_theme(style="whitegrid", palette="Set2")
RANDOM_STATE = 42

In [2]:
# Locate the data (works if the 'Data' folder sits next to the notebook, or inside the unzipped project folder)
for d in ["Data", "data", ".", "PRCP-1016-HeartDieseasePred/Data"]:
    if os.path.exists(os.path.join(d, "values.csv")):
        DATA_DIR = d; break
else:
    raise FileNotFoundError("values.csv / labels.csv not found - put the unzipped Data folder next to this notebook")
values = pd.read_csv(os.path.join(DATA_DIR, "values.csv"))
labels = pd.read_csv(os.path.join(DATA_DIR, "labels.csv"))
print("values:", values.shape, "| labels:", labels.shape)
df = values.merge(labels, on="patient_id", how="inner")
print("merged:", df.shape)
df.head()

values: (180, 14) | labels: (180, 2)
merged: (180, 15)


,patient_id,slope_of_peak_exercise_st_segment,thal,resting_blood_pressure,chest_pain_type,num_major_vessels,fasting_blood_sugar_gt_120_mg_per_dl,resting_ekg_results,serum_cholesterol_mg_per_dl,oldpeak_eq_st_depression,sex,age,max_heart_rate_achieved,exercise_induced_angina,heart_disease_present
0,0z64un,1,normal,128,2,0,0,2,308,0.0,1,45,170,0,0
1,ryoo3j,2,normal,110,3,0,0,0,214,1.6,0,54,158,0,0
2,yt1s1x,1,normal,125,4,3,0,2,304,0.0,1,77,162,1,1
3,l2xjde,1,reversible_defect,152,4,0,0,0,223,0.0,1,40,181,0,1
4,oyt4ek,3,reversible_defect,178,1,0,0,2,270,4.2,1,59,145,0,0


Dataset size and types. `thal` is the only text column; everything else is numeric. Several integer columns (`chest_pain_type`, `resting_ekg_results`, `slope_of_peak_exercise_st_segment`, `num_major_vessels`) are really **categories**, not quantities, and are treated that way later.

In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 180 entries, 0 to 179
Data columns (total 15 columns):
 #   Column                                Non-Null Count  Dtype  
---  ------                                --------------  -----  
 0   patient_id                            180 non-null    object 
 1   slope_of_peak_exercise_st_segment     180 non-null    int64  
 2   thal                                  180 non-null    object 
 3   resting_blood_pressure                180 non-null    int64  
 4   chest_pain_type                       180 non-null    int64  
 5   num_major_vessels                     180 non-null    int64  
 6   fasting_blood_sugar_gt_120_mg_per_dl  180 non-null    int64  
 7   resting_ekg_results                   180 non-null    int64  
 8   serum_cholesterol_mg_per_dl           180 non-null    int64  
 9   oldpeak_eq_st_depression              180 non-null    float64
 10  sex                                   180 non-null    int64  
 11  age                

**Data quality check:** 180 patients, 13 features, **no missing values and no duplicates**, so no imputation or de-duplication is needed. Value ranges are all clinically plausible (age 29-77, resting BP 94-180, cholesterol 126-564).

In [4]:
print("Missing values:\n", df.isna().sum().sum())
print("Duplicate rows:", df.duplicated().sum(), "| duplicate patient_ids:", df.patient_id.duplicated().sum())
df.describe().T.round(2)

Missing values:
 0
Duplicate rows: 0 | duplicate patient_ids: 0


,count,mean,std,min,25%,50%,75%,max
slope_of_peak_exercise_st_segment,180.0,1.55,0.62,1.0,1.00,1.0,2.00,3.0
resting_blood_pressure,180.0,131.31,17.01,94.0,120.00,130.0,140.00,180.0
chest_pain_type,180.0,3.16,0.94,1.0,3.00,3.0,4.00,4.0
num_major_vessels,180.0,0.69,0.97,0.0,0.00,0.0,1.00,3.0
fasting_blood_sugar_gt_120_mg_per_dl,180.0,0.16,0.37,0.0,0.00,0.0,0.00,1.0
resting_ekg_results,180.0,1.05,1.00,0.0,0.00,2.0,2.00,2.0
serum_cholesterol_mg_per_dl,180.0,249.21,52.72,126.0,213.75,245.5,281.25,564.0
oldpeak_eq_st_depression,180.0,1.01,1.12,0.0,0.00,0.8,1.60,6.2
sex,180.0,0.69,0.46,0.0,0.00,1.0,1.00,1.0
age,180.0,54.81,9.33,29.0,48.00,55.0,62.00,77.0


## 2. Exploratory data analysis (Task 1)
### 2.1 Target balance
The classes are fairly balanced (55.6% healthy vs 44.4% disease), so no resampling (SMOTE etc.) is required. A stratified split is still used to keep this ratio in every fold.

In [5]:
TARGET = "heart_disease_present"
df = df.drop(columns="patient_id")   # random identifier, no predictive meaning
num_cols = ["age","resting_blood_pressure","serum_cholesterol_mg_per_dl","max_heart_rate_achieved","oldpeak_eq_st_depression"]
cat_cols = ["thal","chest_pain_type","slope_of_peak_exercise_st_segment","resting_ekg_results","num_major_vessels"]
bin_cols = ["sex","fasting_blood_sugar_gt_120_mg_per_dl","exercise_induced_angina"]
counts = df[TARGET].value_counts()
print(counts, "\n", (counts/len(df)*100).round(1))
fig, ax = plt.subplots(figsize=(4.5,3.5))
sns.countplot(x=TARGET, data=df, ax=ax); ax.set_xticklabels(["No disease (0)","Disease (1)"]); ax.set_title("Target distribution")
plt.tight_layout(); plt.show()

heart_disease_present
0    100
1     80
Name: count, dtype: int64 
 heart_disease_present
0    55.6
1    44.4
Name: count, dtype: float64


### 2.2 Numeric features vs. the target

In [6]:
fig, axes = plt.subplots(2,3, figsize=(14,7))
for ax, c in zip(axes.ravel(), num_cols):
    sns.histplot(data=df, x=c, hue=TARGET, kde=True, ax=ax, element="step")
    ax.set_title(c)
axes.ravel()[-1].axis("off")
plt.tight_layout(); plt.show()

**Reading the plots:** patients with disease reach a **lower maximum heart rate** and show **higher ST depression (oldpeak)**. Cholesterol and resting blood pressure overlap heavily between the two groups, so on their own they separate patients poorly.

In [7]:
fig, axes = plt.subplots(1,5, figsize=(18,4))
for ax, c in zip(axes, num_cols):
    sns.boxplot(data=df, x=TARGET, y=c, ax=ax); ax.set_title(c)
plt.tight_layout(); plt.show()

Outliers (IQR rule) are few: 6 in blood pressure, 2 in cholesterol, 4 in oldpeak. They are medically plausible values (e.g. a very high cholesterol or a deep ST depression is exactly what a sick patient can show), so they are **kept**, and the scaling + regularised models handle them.

In [8]:
# Outlier count by the IQR rule
rows=[]
for c in num_cols:
    q1,q3 = df[c].quantile([.25,.75]); iqr=q3-q1
    lo,hi = q1-1.5*iqr, q3+1.5*iqr
    rows.append((c, int(((df[c]<lo)|(df[c]>hi)).sum()), round(lo,1), round(hi,1)))
pd.DataFrame(rows, columns=["feature","n_outliers","lower_fence","upper_fence"])

,feature,n_outliers,lower_fence,upper_fence
0,age,0,27.0,83.0
1,resting_blood_pressure,6,90.0,170.0
2,serum_cholesterol_mg_per_dl,2,112.5,382.5
3,max_heart_rate_achieved,0,80.6,217.6
4,oldpeak_eq_st_depression,4,-2.4,4.0


### 2.3 Categorical / binary features vs. the target

In [9]:
fig, axes = plt.subplots(2,4, figsize=(18,8))
for ax, c in zip(axes.ravel(), cat_cols+bin_cols):
    sns.countplot(data=df, x=c, hue=TARGET, ax=ax); ax.set_title(c); ax.tick_params(axis="x", rotation=20)
plt.tight_layout(); plt.show()

**Key findings (disease rate per group):**
- **thal:** normal 20% vs reversible defect 76% (fixed defect 50%, only 8 patients).
- **chest pain type 4** (asymptomatic): 72%, versus 14-31% for types 1-3.
- **exercise-induced angina:** 77% vs 29% without.
- **number of major vessels:** rises steadily 27% (0 vessels) -> 59% -> 74% -> 86% (3 vessels).
- **sex:** males 56% vs females 20% (the sample is 69% male, so this is partly a sampling effect).
- **fasting blood sugar > 120:** 45% vs 44%, so **no signal** at all.

In [10]:
# Disease rate for each category value
for c in cat_cols+bin_cols:
    print(f"\n{c}"); print((df.groupby(c)[TARGET].agg(["mean","count"]).rename(columns={"mean":"disease_rate"})).round(2))


thal
                   disease_rate  count
thal                                  
fixed_defect               0.50      8
normal                     0.20     98
reversible_defect          0.76     74

chest_pain_type
                 disease_rate  count
chest_pain_type                     
1                        0.31     13
2                        0.14     28
3                        0.23     57
4                        0.72     82

slope_of_peak_exercise_st_segment
                                   disease_rate  count
slope_of_peak_exercise_st_segment                     
1                                          0.27     93
2                                          0.63     75
3                                          0.67     12

resting_ekg_results
                     disease_rate  count
resting_ekg_results                     
0                            0.36     85
1                            1.00      1
2                            0.51     94

num_major_vessels
     

### 2.4 Statistical significance
Chi-square tests (categorical) and Mann-Whitney U tests (numeric, non-normal data) confirm the picture: `thal`, `chest_pain_type`, `exercise_induced_angina`, `num_major_vessels`, `oldpeak`, `max_heart_rate`, `sex` and `slope` are highly significant (p < 0.001). `age` is borderline (p ~ 0.04). `resting_ekg_results`, `cholesterol`, `resting_blood_pressure` and `fasting_blood_sugar` are **not** significant on their own.

In [11]:
# Statistical tests: chi-square for categorical/binary features, Mann-Whitney U for numeric features
res=[]
for c in cat_cols+bin_cols:
    chi2,p,_,_ = stats.chi2_contingency(pd.crosstab(df[c], df[TARGET]))
    res.append((c,"chi-square",round(p,5)))
for c in num_cols:
    u,p = stats.mannwhitneyu(df.loc[df[TARGET]==1,c], df.loc[df[TARGET]==0,c])
    res.append((c,"Mann-Whitney U",round(p,5)))
tests = pd.DataFrame(res, columns=["feature","test","p_value"]).sort_values("p_value")
tests["significant (p<0.05)"] = tests.p_value<0.05
tests

,feature,test,p_value,significant (p<0.05)
0,thal,chi-square,0.00000,True
1,chest_pain_type,chi-square,0.00000,True
4,num_major_vessels,chi-square,0.00000,True
7,exercise_induced_angina,chi-square,0.00000,True
11,max_heart_rate_achieved,Mann-Whitney U,0.00000,True
12,oldpeak_eq_st_depression,Mann-Whitney U,0.00000,True
2,slope_of_peak_exercise_st_segment,chi-square,0.00001,True
5,sex,chi-square,0.00001,True
8,age,Mann-Whitney U,0.04344,True
3,resting_ekg_results,chi-square,0.07781,False


### 2.5 Correlation analysis
The strongest correlations with the target are `thal` (normal -0.53 / reversible defect +0.53), exercise-induced angina (0.45), number of major vessels (0.42), chest pain type (0.41), oldpeak (0.38) and max heart rate (-0.38). No pair of features is so strongly correlated (multicollinearity) that it needs removing.

In [12]:
corr_df = pd.get_dummies(df, columns=["thal"], drop_first=False).astype(float)
plt.figure(figsize=(12,9))
sns.heatmap(corr_df.corr(), annot=True, fmt=".2f", cmap="coolwarm", center=0, annot_kws={"size":7})
plt.title("Correlation heatmap"); plt.tight_layout(); plt.show()
corr_df.corr()[TARGET].drop(TARGET).sort_values(key=abs, ascending=False).round(3)

thal_normal                            -0.529
thal_reversible_defect                  0.525
exercise_induced_angina                 0.449
num_major_vessels                       0.422
chest_pain_type                         0.413
oldpeak_eq_st_depression                0.383
max_heart_rate_achieved                -0.375
slope_of_peak_exercise_st_segment       0.344
sex                                     0.335
resting_ekg_results                     0.146
age                                     0.138
serum_cholesterol_mg_per_dl             0.080
resting_blood_pressure                  0.079
thal_fixed_defect                       0.024
fasting_blood_sugar_gt_120_mg_per_dl    0.003
Name: heart_disease_present, dtype: float64

Pair plot of the numeric features, coloured by outcome.

In [13]:
sns.pairplot(df[num_cols+[TARGET]], hue=TARGET, corner=True, height=2)
plt.show()

## 3. Preprocessing
- Drop `patient_id` (random identifier, no predictive meaning).
- **Stratified 80/20 split** (144 train / 36 test) - the test set is never touched until the final check.
- **Numeric** features -> `StandardScaler` (needed for Logistic Regression, KNN, SVM).
- **Nominal categories** (`thal`, `chest_pain_type`, `resting_ekg_results`, `slope`) -> one-hot encoding.
- Binary features and `num_major_vessels` (already ordered 0-3) are passed through.
- Everything lives inside a scikit-learn `Pipeline`, so scaling/encoding is learned **only on training folds** during cross-validation (no data leakage).

In [14]:
# Preprocessing + split
X = df.drop(columns=TARGET); y = df[TARGET]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE)
print(X_train.shape, X_test.shape, y_train.mean().round(3), y_test.mean().round(3))
onehot_cols = ["thal","chest_pain_type","slope_of_peak_exercise_st_segment","resting_ekg_results"]
passthrough = bin_cols + ["num_major_vessels"]
preprocess = ColumnTransformer([
    ("num", StandardScaler(), num_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore", drop="first"), onehot_cols),
    ("pass", "passthrough", passthrough)])

(144, 13) (36, 13) 0.444 0.444


## 4. Model building (Task 2)
### 4.1 Baseline comparison of 9 algorithms
With only 144 training rows, a single train/validation split would be very noisy. Each model is therefore scored with **repeated stratified 5-fold cross-validation (5 x 5 = 25 fits)**.

In [15]:
models = {
 "Logistic Regression": LogisticRegression(max_iter=2000, random_state=RANDOM_STATE),
 "KNN": KNeighborsClassifier(n_neighbors=7),
 "SVM (RBF)": SVC(probability=True, random_state=RANDOM_STATE),
 "Naive Bayes": GaussianNB(),
 "Decision Tree": DecisionTreeClassifier(max_depth=4, random_state=RANDOM_STATE),
 "Random Forest": RandomForestClassifier(n_estimators=300, random_state=RANDOM_STATE),
 "Extra Trees": ExtraTreesClassifier(n_estimators=300, random_state=RANDOM_STATE),
 "AdaBoost": AdaBoostClassifier(random_state=RANDOM_STATE),
 "Gradient Boosting": GradientBoostingClassifier(random_state=RANDOM_STATE),
}
scoring = {"accuracy":"accuracy","precision":"precision","recall":"recall","f1":"f1","roc_auc":"roc_auc"}
cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=5, random_state=RANDOM_STATE)
rows=[]
for name, m in models.items():
    pipe = Pipeline([("prep", preprocess), ("clf", m)])
    s = cross_validate(pipe, X_train, y_train, cv=cv, scoring=scoring, n_jobs=-1)
    rows.append({"Model":name, **{k: s[f"test_{k}"].mean() for k in scoring}, "roc_auc_std": s["test_roc_auc"].std()})
baseline = pd.DataFrame(rows).set_index("Model").sort_values("roc_auc", ascending=False).round(3)
baseline

,accuracy,precision,recall,f1,roc_auc,roc_auc_std
Model,,,,,,
Naive Bayes,0.804,0.782,0.786,0.780,0.865,0.050
Logistic Regression,0.806,0.810,0.753,0.774,0.863,0.058
Random Forest,0.781,0.774,0.738,0.750,0.858,0.048
SVM (RBF),0.781,0.786,0.725,0.746,0.846,0.062
Extra Trees,0.791,0.777,0.766,0.766,0.842,0.060
AdaBoost,0.754,0.752,0.700,0.717,0.827,0.054
KNN,0.754,0.761,0.690,0.711,0.819,0.054
Gradient Boosting,0.743,0.727,0.694,0.704,0.818,0.052
Decision Tree,0.720,0.710,0.647,0.670,0.709,0.082


### 4.2 Hyperparameter tuning
The four strongest families (Logistic Regression, SVM, Random Forest, Gradient Boosting) are tuned with `GridSearchCV`, optimising ROC-AUC. Decision Tree was clearly the weakest and is dropped.

In [16]:
# Hyperparameter tuning of the strongest candidates
grids = {
 "Logistic Regression": (LogisticRegression(max_iter=2000, random_state=RANDOM_STATE),
     {"clf__C":[0.01,0.03,0.1,0.3,1,3,10], "clf__penalty":["l2"]}),
 "SVM (RBF)": (SVC(probability=True, random_state=RANDOM_STATE),
     {"clf__C":[0.1,0.3,1,3,10], "clf__gamma":["scale",0.01,0.03,0.1]}),
 "Random Forest": (RandomForestClassifier(n_estimators=300, random_state=RANDOM_STATE),
     {"clf__max_depth":[2,3,4,6,None], "clf__min_samples_leaf":[1,3,5], "clf__max_features":["sqrt",0.5]}),
 "Gradient Boosting": (GradientBoostingClassifier(random_state=RANDOM_STATE),
     {"clf__n_estimators":[50,100,200], "clf__learning_rate":[0.03,0.1], "clf__max_depth":[1,2,3], "clf__subsample":[0.7,1.0]}),
}
inner_cv = StratifiedKFold(5, shuffle=True, random_state=RANDOM_STATE)
tuned = {}
for name,(m,g) in grids.items():
    gs = GridSearchCV(Pipeline([("prep", preprocess),("clf", m)]), g, cv=inner_cv, scoring="roc_auc", n_jobs=-1)
    gs.fit(X_train, y_train)
    tuned[name]=gs
    print(f"{name:20s} best CV ROC-AUC={gs.best_score_:.3f}  params={gs.best_params_}")

Logistic Regression  best CV ROC-AUC=0.881  params={'clf__C': 0.3, 'clf__penalty': 'l2'}
SVM (RBF)            best CV ROC-AUC=0.876  params={'clf__C': 3, 'clf__gamma': 0.01}
Random Forest        best CV ROC-AUC=0.885  params={'clf__max_depth': 4, 'clf__max_features': 'sqrt', 'clf__min_samples_leaf': 3}
Gradient Boosting    best CV ROC-AUC=0.856  params={'clf__learning_rate': 0.03, 'clf__max_depth': 2, 'clf__n_estimators': 200, 'clf__subsample': 0.7}


### 4.3 Test-set evaluation
Results on the 36 held-out patients. **Caution:** with only 36 rows one patient changes accuracy by about 3 points, so these numbers (including the perfect recall) are optimistic and noisy. Section 7 uses the more reliable cross-validated numbers for the decision.

In [17]:
# Evaluate tuned models on the untouched test set
rows=[]; probas={}
for name, gs in tuned.items():
    p = gs.predict_proba(X_test)[:,1]; pred = (p>=0.5).astype(int); probas[name]=p
    rows.append({"Model":name,"Accuracy":accuracy_score(y_test,pred),"Precision":precision_score(y_test,pred),
                 "Recall":recall_score(y_test,pred),"F1":f1_score(y_test,pred),"ROC-AUC":roc_auc_score(y_test,p)})
test_results = pd.DataFrame(rows).set_index("Model").round(3).sort_values("ROC-AUC", ascending=False)
test_results

,Accuracy,Precision,Recall,F1,ROC-AUC
Model,,,,,
SVM (RBF),0.917,0.842,1.000,0.914,0.969
Gradient Boosting,0.917,0.842,1.000,0.914,0.962
Logistic Regression,0.917,0.842,1.000,0.914,0.959
Random Forest,0.889,0.833,0.938,0.882,0.959


ROC curves on the test set - all tuned models are close together.

In [18]:
fig, ax = plt.subplots(figsize=(6,5))
for name,p in probas.items():
    fpr,tpr,_ = roc_curve(y_test,p); ax.plot(fpr,tpr,label=f"{name} (AUC={roc_auc_score(y_test,p):.2f})")
ax.plot([0,1],[0,1],"k--"); ax.set_xlabel("False positive rate"); ax.set_ylabel("True positive rate"); ax.set_title("ROC curves (test set)"); ax.legend()
plt.tight_layout(); plt.show()

### 4.4 More reliable comparison: cross-validation on all 180 patients
Repeated 5-fold CV (10 repeats) on the full data. The top three models are **statistically tied** (ROC-AUC about 0.89, standard deviation about 0.06).

In [19]:
# Out-of-fold comparison on the FULL data (more stable than a 36-row test set)
rows=[]
cv_full = RepeatedStratifiedKFold(n_splits=5, n_repeats=10, random_state=7)
for name, gs in tuned.items():
    s = cross_validate(gs.best_estimator_, X, y, cv=cv_full, scoring=scoring, n_jobs=-1)
    rows.append({"Model":name, **{k: s[f"test_{k}"].mean() for k in scoring}, "roc_auc_std": s["test_roc_auc"].std()})
final_cv = pd.DataFrame(rows).set_index("Model").sort_values("roc_auc", ascending=False).round(3)
final_cv

,accuracy,precision,recall,f1,roc_auc,roc_auc_std
Model,,,,,,
SVM (RBF),0.838,0.855,0.770,0.806,0.893,0.057
Logistic Regression,0.845,0.852,0.794,0.817,0.892,0.057
Random Forest,0.830,0.828,0.788,0.804,0.891,0.057
Gradient Boosting,0.800,0.794,0.755,0.769,0.868,0.061


## 5. Final model selection and evaluation
Logistic Regression is chosen (reasoning in Section 7).

In [20]:
# SVM, Logistic Regression and Random Forest are statistically tied (ROC-AUC 0.89 +/- 0.06).
# We pick Logistic Regression: same accuracy, fully interpretable (odds ratios), fast, easy to audit - important in healthcare.
best_name = "Logistic Regression"; best_model = tuned[best_name].best_estimator_
print("Selected model:", best_name)
p = best_model.predict_proba(X_test)[:,1]
print(classification_report(y_test, (p>=0.5).astype(int), target_names=["No disease","Disease"]))
ConfusionMatrixDisplay.from_predictions(y_test,(p>=0.5).astype(int), display_labels=["No disease","Disease"], cmap="Blues"); plt.show()

Selected model: Logistic Regression
              precision    recall  f1-score   support

  No disease       1.00      0.85      0.92        20
     Disease       0.84      1.00      0.91        16

    accuracy                           0.92        36
   macro avg       0.92      0.93      0.92        36
weighted avg       0.93      0.92      0.92        36



### 5.1 Decision threshold (recall vs precision)
Lowering the threshold from 0.5 to about 0.35-0.40 catches more sick patients at the cost of more false alarms. For screening this is usually a good trade (see Section 8).

In [21]:
# Threshold analysis using out-of-fold probabilities on all data (recall vs precision trade-off)
oof = cross_val_predict(best_model, X, y, cv=StratifiedKFold(5, shuffle=True, random_state=RANDOM_STATE), method="predict_proba")[:,1]
rows=[]
for t in [0.3,0.35,0.4,0.45,0.5,0.55,0.6]:
    pr=(oof>=t).astype(int)
    rows.append((t, recall_score(y,pr), precision_score(y,pr), f1_score(y,pr), int(((pr==0)&(y==1)).sum())))
thr = pd.DataFrame(rows, columns=["threshold","recall","precision","f1","missed_patients"]).round(3); thr

,threshold,recall,precision,f1,missed_patients
0,0.30,0.875,0.714,0.787,10
1,0.35,0.862,0.750,0.802,11
2,0.40,0.825,0.786,0.805,14
3,0.45,0.812,0.823,0.818,15
4,0.50,0.775,0.886,0.827,18
5,0.55,0.762,0.897,0.824,19
6,0.60,0.712,0.891,0.792,23


## 6. Feature importance
Permutation importance (drop in ROC-AUC when a feature is shuffled). `thal` and `num_major_vessels` are the most important, followed by `chest_pain_type`. Fasting blood sugar adds nothing.

In [22]:
# Feature importance via permutation importance (works for any model)
r = permutation_importance(best_model, X_test, y_test, n_repeats=30, scoring="roc_auc", random_state=RANDOM_STATE)
imp = pd.Series(r.importances_mean, index=X.columns).sort_values()
r_full = permutation_importance(best_model.fit(X_train,y_train), X, y, n_repeats=30, scoring="roc_auc", random_state=RANDOM_STATE)
imp_full = pd.Series(r_full.importances_mean, index=X.columns).sort_values()
imp_full.plot.barh(figsize=(7,5), title="Permutation importance (drop in ROC-AUC)"); plt.tight_layout(); plt.show()
imp_full.sort_values(ascending=False).round(3)

num_major_vessels                       0.045
thal                                    0.045
chest_pain_type                         0.026
sex                                     0.015
oldpeak_eq_st_depression                0.012
resting_ekg_results                     0.008
max_heart_rate_achieved                 0.007
exercise_induced_angina                 0.004
serum_cholesterol_mg_per_dl             0.004
age                                     0.003
slope_of_peak_exercise_st_segment       0.003
resting_blood_pressure                  0.002
fasting_blood_sugar_gt_120_mg_per_dl   -0.001
dtype: float64

**Odds ratios** from Logistic Regression (above 1 raises risk, below 1 lowers it; numeric features are per 1 standard deviation). Examples: reversible thal defect about 2x, chest pain type 4 about 2x, each extra blocked vessel about 1.9x; a high max heart rate (0.71) and normal thal (0.54) are protective.

In [23]:
# Interpretability: odds ratios from the Logistic Regression
lr = tuned["Logistic Regression"].best_estimator_.fit(X_train, y_train)
names = lr.named_steps["prep"].get_feature_names_out()
odds = pd.Series(np.exp(lr.named_steps["clf"].coef_[0]), index=[n.split("__")[1] for n in names]).sort_values(ascending=False).round(2)
odds.rename("odds_ratio").to_frame()

,odds_ratio
chest_pain_type_4,2.05
sex,2.05
thal_reversible_defect,2.04
num_major_vessels,1.93
resting_ekg_results_2,1.62
exercise_induced_angina,1.60
oldpeak_eq_st_depression,1.45
slope_of_peak_exercise_st_segment_2,1.35
slope_of_peak_exercise_st_segment_3,1.21
serum_cholesterol_mg_per_dl,1.19


### Save the final model
The selected pipeline is refit on all 180 patients and saved so it can be loaded for predictions with `joblib.load`.

In [24]:
# Save model
import joblib
final_model = tuned[best_name].best_estimator_.fit(X, y)
joblib.dump(final_model, "heart_disease_model.joblib")
print("Saved heart_disease_model.joblib")

Saved heart_disease_model.joblib


## 7. Model Comparison Report

**Cross-validated results on all 180 patients (tuned models, repeated 5-fold, mean):**

| Model | Accuracy | Precision | Recall | F1 | ROC-AUC |
|---|---|---|---|---|---|
| SVM (RBF) | 0.838 | 0.855 | 0.770 | 0.806 | 0.893 |
| **Logistic Regression** | **0.845** | 0.852 | **0.794** | **0.817** | 0.892 |
| Random Forest | 0.830 | 0.828 | 0.788 | 0.804 | 0.891 |
| Gradient Boosting | 0.802 | 0.796 | 0.759 | 0.771 | 0.869 |

Untuned baselines for reference (ROC-AUC): Naive Bayes 0.865, Logistic Regression 0.863, Random Forest 0.858, SVM 0.846, Extra Trees 0.842, AdaBoost 0.827, KNN 0.819, Gradient Boosting 0.817, Decision Tree 0.709.

**Observations**
- The three best models are within 0.002 ROC-AUC of each other, far less than the 0.06 fold-to-fold standard deviation, so there is **no real performance winner**.
- A single decision tree overfits (0.71 AUC); boosting did not help on such a small dataset.
- The held-out test set (36 patients) gave about 92% accuracy and 100% recall for LR/SVM/GB, but this is a lucky small sample; expect roughly **84% accuracy, 79% recall, 0.89 AUC** in practice.

**Recommendation for production: Logistic Regression** (tuned C = 0.3), because
1. performance equals the best model;
2. it is **interpretable** (odds ratios), which clinicians and regulators expect;
3. it is fast, stable, well calibrated and easy to monitor and retrain;
4. with 180 rows, more complex models offer no gain and carry more overfitting risk.

Use a **decision threshold of about 0.35-0.40** instead of 0.5 to prioritise recall (about 83-86% instead of 78%).

## 8. Suggestions to the hospital (Task 3)

**1. Use the model as a screening and triage aid, not a diagnosis.** Run it on routine check-up data and flag patients above the threshold for cardiologist review. Final decisions stay with doctors.

**2. Focus on the strongest warning signs.** The data points to: abnormal thallium stress test (reversible defect), more blocked major vessels on fluoroscopy, asymptomatic chest pain (type 4), exercise-induced angina, high ST depression and a low maximum heart rate. Patients with several of these together should be fast-tracked.

**3. Do not be reassured by "no chest pain".** Chest pain type 4 (asymptomatic) had the *highest* disease rate (72%). Screen patients with other risk factors even if they have no symptoms.

**4. Tune the alert threshold to the goal.** At 0.5 the model misses about 18 of 80 sick patients in cross-validation; at 0.35 it misses about 11, with more false alarms (precision 75%). For screening, choose the lower threshold; for limited-capacity clinics, a middle value (0.40-0.45).

**5. Early-warning workflow.** Integrate the model into the patient record so risk is scored automatically after stress-test/ECG results are entered; send high-risk patients for angiography or specialist referral, and moderate-risk ones for lifestyle and follow-up programmes.

**6. Prevention programmes.** Target men over 50 and patients with exertional angina for blood pressure, cholesterol, exercise and smoking-cessation counselling.

**7. Do not rely on fasting blood sugar alone.** It showed no link to disease in this sample, although diabetes remains a risk factor in medical literature; keep it in the routine panel but do not use it as a stand-alone alarm.

**8. Improve the data and the model over time.** Collect more patients (especially women, and younger/older groups), add smoking, family history, BMI and lipid profile, validate on data from your own hospital, monitor performance and recalibrate regularly, and check fairness across sex and age groups.

**Limitations.** Only 180 patients, 69% male, one data source; results must be validated on local data before clinical use.

## 9. Report on challenges faced

| # | Challenge | Technique used | Reason |
|---|---|---|---|
| 1 | Data split across two files (`values`, `labels`) | Merge on `patient_id`, then drop the id | Join gives one table; an id carries no medical information and would add noise |
| 2 | **Very small dataset (180 rows)** - a single test split is unreliable (1 patient = ~3% accuracy) | Repeated stratified k-fold CV (5x5, 10x5) and a stratified split | Averages over many splits, giving stable estimates and a measure of spread |
| 3 | Risk of overfitting with flexible models | Regularised/simple models, limited tree depth, tuned with GridSearch inside CV | Keeps model complexity matched to data size; decision tree overfit (AUC 0.71) was spotted this way |
| 4 | Numbers that are really categories (`chest_pain_type`, `slope`, `ekg`) and a text column (`thal`) | One-hot encoding for nominal values, `num_major_vessels` kept ordinal | Prevents models from assuming a false numeric order (e.g. pain type 4 = 4x type 1) |
| 5 | Features on very different scales (cholesterol ~250 vs oldpeak ~1) | `StandardScaler` for numeric columns | Required for distance/margin models (KNN, SVM, LR) and for comparable coefficients |
| 6 | Outliers (BP, cholesterol, oldpeak) | Detected with IQR, **kept** | Values are medically plausible; removing them would delete real sick patients from a small dataset |
| 7 | Rare categories (fixed_defect n=8, ekg value 1 has n=1) | Kept; `handle_unknown="ignore"` in encoder; noted as limitation | Dropping loses information; ignoring unseen values avoids crashes in production |
| 8 | Data leakage risk (scaling before splitting) | `Pipeline` + `ColumnTransformer` | Preprocessing is fit only on training folds |
| 9 | Several models nearly tied | Compared on 5 metrics with std. dev.; chose by interpretability and robustness | Differences were smaller than noise, so practical factors decide |
| 10 | Cost of errors is not symmetric (missing a sick patient is worse) | Focus on recall + ROC-AUC, threshold analysis | Accuracy alone hides missed patients; lowering the threshold trades precision for recall |
| 11 | Class balance | Checked (44% / 56%); no resampling | Balanced enough; resampling on so few rows would add artificial noise |
| 12 | Sample bias (69% male, one source) | Flagged in findings and recommendations | Prevents over-claiming; calls for local validation |

## 10. Conclusion
A Logistic Regression model reaches about **0.89 ROC-AUC, 84% accuracy and 79% recall** (cross-validated), identifying thal result, number of blocked vessels, chest pain type, exercise angina, ST depression and heart rate as the main risk signals. With a lower decision threshold it can serve as a practical early-warning tool, provided it is validated on larger, local data first.